# LLM Agent Decision Analysis

This notebook reads the decision CSVs generated by the LLM benchmark scripts for **GPT-4o mini**, **GPT-5 nano**, **GPT-5 mini**, **Qwen Turbo**, and **Qwen3.7 Flash**, including both goal and no-goal treatments. Multi-model figures use this same top-to-bottom order.

**Metrics computed:**
1. P(reproduce | free_cells)
2. P(reproduce | energy)
3. Heatmap P(reproduce | energy × free_cells)
4. Number and share of movement choices consistent with the rule at alpha = 1
5. P(Stay | food_here)
6. Action distribution by energy level
7. Reproduction threshold estimation


In [ ]:
from pathlib import Path
import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

from IPython.display import display, Markdown

warnings.filterwarnings('ignore', category=FutureWarning)

# ── LaTeX-style plot configuration (matches paper_benchmark_analysis) ──
sns.set_theme(style='ticks', context='notebook')
plt.rcParams.update({
    'font.size': 12,
    'font.family': 'serif',
    'mathtext.fontset': 'cm',
    'figure.dpi': 110,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 14,
    'figure.titleweight': 'bold',
})

# ── Paths ──
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / 'run_analysis.py').exists():
            return candidate
    raise FileNotFoundError('Could not find the project folder.')

PROJECT_ROOT = find_project_root()
RESULTS_DIR = PROJECT_ROOT / 'results'
LLM_MODELS = [m.strip() for m in os.getenv('LLM_MODELS', 'gpt-4o-mini,gpt-5-nano,gpt-5-mini,qwen-turbo,qwen3.7-flash-2026-07-15').split(',') if m.strip()]
MODEL_LABELS = {
    'gpt-4o-mini': 'GPT-4o mini',
    'gpt-5-mini': 'GPT-5 mini',
    'gpt-5-nano': 'GPT-5 nano',
    'qwen-turbo': 'Qwen Turbo',
    'qwen3.7-flash-2026-07-15': 'Qwen3.7 Flash',
}
def model_slug(model):
    return re.sub(r'[^A-Za-z0-9._-]+', '-', model).strip('-._') or 'unknown-model'
DECISION_FILES = {model: {
    'Goal stated': RESULTS_DIR / f'llm_decisions_{model_slug(model)}.csv',
    'No stated goal': RESULTS_DIR / f'llm_decisions_no_goal_{model_slug(model)}.csv',
} for model in LLM_MODELS}

print(f'Project root: {PROJECT_ROOT}')
for model, files in DECISION_FILES.items():
    print(f'{MODEL_LABELS.get(model, model)}:')
    for variant, path in files.items():
        print(f'  {variant}: {path}')


In [ ]:
# Load every available model/treatment pair. Missing GPT-5 mini files are reported
# but do not prevent inspecting the existing GPT-4o mini benchmark.
frames = []
for model in LLM_MODELS:
    for prompt_variant, path in DECISION_FILES[model].items():
        if not path.exists():
            print(f'Missing: {path.name}')
            continue
        part = pd.read_csv(path)
        part['model'] = model
        part['model_label'] = MODEL_LABELS.get(model, model)
        part['prompt_variant'] = prompt_variant
        frames.append(part)
if not frames:
    raise FileNotFoundError('No LLM decision CSV found. Run `python run_gpt5_mini_benchmarks.py` and/or `python run_analysis_llm.py`.')
df = pd.concat(frames, ignore_index=True)
MODEL_ORDER = [MODEL_LABELS.get(m, m) for m in LLM_MODELS if MODEL_LABELS.get(m, m) in df['model_label'].unique()]
PROMPT_ORDER = [v for v in ['Goal stated', 'No stated goal'] if v in df['prompt_variant'].unique()]

# Ensure correct types
bool_cols = ['reproduce', 'did_reproduce', 'did_move']
for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.lower().isin({'true', '1', 'yes'})

int_cols = ['episode', 'step', 'occ_N', 'occ_S', 'occ_E', 'occ_W', 'free_cells']
for col in int_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

float_cols = ['food_N', 'food_S', 'food_E', 'food_W', 'food_here', 'energy']
for col in float_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'Total decisions: {len(df):,}')
print(f'Scenarios: {sorted(df["scenario"].unique())}')
print(f'Episodes: {df.groupby("scenario")["episode"].nunique().to_dict()}')
print(f'Columns: {list(df.columns)}')
display(df.head(10))


In [ ]:
# Colour palette for scenarios
# Discrete samples from plasma: enough separation for five categorical series.
PLASMA = mpl.colormaps['plasma']
SCENARIO_LEVELS = ['S1', 'S2', 'S3', 'S4', 'S5']
SCENARIO_PALETTE = dict(zip(
    SCENARIO_LEVELS,
    [PLASMA(x) for x in np.linspace(0.08, 0.92, len(SCENARIO_LEVELS))],
))

ACTION_NAMES = ['N', 'S', 'E', 'W', 'Stay']
ACTION_COLORS = dict(zip(
    ACTION_NAMES,
    [PLASMA(x) for x in np.linspace(0.08, 0.92, len(ACTION_NAMES))],
))
FREE_CELL_COLORS = dict(zip(
    range(5),
    [PLASMA(x) for x in np.linspace(0.08, 0.92, 5)],
))
PROMPT_COLORS = {'Goal stated': '#B23A48', 'No stated goal': '#F08A95'}

# Fixed bins make every energy-based chart directly comparable.
ENERGY_BINS = [0.0, 1.5, 3.0, 4.5, 6.0, np.inf]
ENERGY_LABELS = ['0–1.5', '1.5–3.0', '3.0–4.5', '4.5–6.0', '6.0+']

# Transfer cost used by the rule-based movement threshold.
MOVE_COST_FRACTION = 0.20

def energy_bins(series):
    return pd.cut(series, bins=ENERGY_BINS, labels=ENERGY_LABELS, right=False, include_lowest=True)


## 1. Reproduction probability vs. Free neighbouring cells

$P(\text{reproduce} \mid \text{free\_cells} = k)$ for $k = 0, 1, 2, 3, 4$.

This answers: **does the LLM understand that free cells are needed to reproduce?**


In [ ]:
repro_fc = df.groupby(['model_label', 'prompt_variant', 'free_cells'])['did_reproduce'].agg(['mean', 'count']).reset_index()
repro_fc.columns = ['model_label', 'prompt_variant', 'free_cells', 'P_reproduce', 'n']

fig, axes = plt.subplots(len(MODEL_ORDER), 1, figsize=(8, 4.8 * len(MODEL_ORDER)), squeeze=False, sharex=True, sharey=True)
for row, model_label in enumerate(MODEL_ORDER):
    ax = axes[row, 0]
    sns.barplot(data=repro_fc[repro_fc['model_label'] == model_label], x='free_cells', y='P_reproduce',
                hue='prompt_variant', hue_order=PROMPT_ORDER, palette=PROMPT_COLORS, edgecolor='white', ax=ax)
    ax.set_xlabel('Free neighbouring cells')
    ax.set_ylabel(r'$P(\mathrm{reproduce})$')
    ax.set_title(f'{model_label}: ' + r'$P(\mathrm{reproduce} \mid \mathrm{free\_cells} = k)$')
    ax.set_xticks(range(5))
    ax.legend(title='System prompt', frameon=False)
    sns.despine(ax=ax)
plt.tight_layout()
plt.show()

display(repro_fc.style.format({'P_reproduce': '{:.4f}', 'n': '{:,}'}))


## 2. Reproduction probability vs. Energy

$P(\text{reproduce} \mid E \in \text{bin})$

This answers: **does the LLM save energy before reproducing?**


In [ ]:
df['energy_bin'] = energy_bins(df['energy'])
repro_energy = df.groupby(['model_label', 'prompt_variant', 'energy_bin'], observed=True)['did_reproduce'].agg(['mean', 'count']).reset_index()
repro_energy.columns = ['model_label', 'prompt_variant', 'energy_bin', 'P_reproduce', 'n']

fig, axes = plt.subplots(len(MODEL_ORDER), 1, figsize=(10, 4.8 * len(MODEL_ORDER)), squeeze=False, sharex=True, sharey=True)
x = np.arange(len(ENERGY_LABELS))
for row, model_label in enumerate(MODEL_ORDER):
    ax = axes[row, 0]
    model_data = repro_energy[repro_energy['model_label'] == model_label]
    for variant in PROMPT_ORDER:
        subset = model_data[model_data['prompt_variant'] == variant].set_index('energy_bin').reindex(ENERGY_LABELS)
        ax.plot(x, subset['P_reproduce'], 'o-', label=variant, color=PROMPT_COLORS[variant], markersize=7, linewidth=2)
    ax.set_xlabel('Agent energy')
    ax.set_xticks(x)
    ax.set_xticklabels(ENERGY_LABELS)
    ax.set_ylabel(r'$P(\mathrm{reproduce})$')
    ax.set_title(f'{model_label}: ' + r'$P(\mathrm{reproduce} \mid E \in \mathrm{bin})$')
    ax.legend(title='System prompt', frameon=False)
    sns.despine(ax=ax)
plt.tight_layout()
plt.show()

display(repro_energy[['energy_bin', 'P_reproduce', 'n']].style.format({'P_reproduce': '{:.4f}', 'n': '{:,}'}))
df.drop(columns='energy_bin', inplace=True)


## 3. Heatmap: $P(\text{reproduce} \mid E, \text{free\_cells})$

Joint effect of energy and free cells on reproduction decisions.


In [ ]:
df['energy_bin'] = energy_bins(df['energy'])

heatmap_df = df.groupby(['model_label', 'prompt_variant', 'energy_bin', 'free_cells'], observed=True)['did_reproduce'].agg(['mean', 'count']).reset_index()
heatmap_df.columns = ['model_label', 'prompt_variant', 'energy_bin', 'free_cells', 'P_reproduce', 'n']

fig, axes = plt.subplots(len(MODEL_ORDER), len(PROMPT_ORDER),
                         figsize=(8 * len(PROMPT_ORDER), 5.5 * len(MODEL_ORDER)), squeeze=False)
for row, model_label in enumerate(MODEL_ORDER):
  for col, variant in enumerate(PROMPT_ORDER):
    ax = axes[row, col]
    panel_data = heatmap_df[(heatmap_df['model_label'] == model_label) & (heatmap_df['prompt_variant'] == variant)]
    if panel_data.empty:
        ax.set_axis_off()
        ax.set_title(f'{model_label} — {variant} (no data)')
        continue
    pivot = 100 * panel_data.pivot(index='energy_bin', columns='free_cells', values='P_reproduce')
    annotations = pivot.map(lambda value: '' if pd.isna(value) else f'{value:.1f}%')
    sns.heatmap(pivot, annot=annotations, fmt='', cmap='YlOrRd', linewidths=0.5,
                vmin=0, vmax=100, cbar=col == len(PROMPT_ORDER) - 1,
                cbar_kws={'label': r'$P(\mathrm{reproduce})$ (\%)'}, ax=ax)
    ax.set_xlabel('Free cells')
    ax.set_ylabel('Energy bin' if col == 0 else '')
    ax.set_title(f'{model_label} — {variant}')
plt.suptitle(r'$P(\mathrm{reproduce} \mid E_{\mathrm{bin}},\, \mathrm{free\_cells})$', y=1.02)
plt.tight_layout()
plt.show()

df.drop(columns='energy_bin', inplace=True)


## 4. Energy-rational movement choices

A movement choice is classified as **rational** when it matches the rule-based policy at $\alpha=1$ (the rational cost-benefit threshold):

$$f_{max} \alpha > f_h + E \cdot ft, \qquad \alpha=1.$$

The rational action is a move to any free neighbour attaining $f_{max}$ when the strict inequality holds; otherwise it is **Stay**. If no neighbouring cell is free, **Stay** is rational. Reproduction is evaluated separately and does not affect this movement classification.


In [ ]:
rational = df.copy()
food_cols = ['food_N', 'food_S', 'food_E', 'food_W']
occ_cols = ['occ_N', 'occ_S', 'occ_E', 'occ_W']
dir_names = ['N', 'S', 'E', 'W']

# Exact RuleBasedAgent decision rule at alpha=1.
ALPHA_RATIONAL = 1.0
food_matrix = rational[food_cols].to_numpy(dtype=float)
occupied = rational[occ_cols].to_numpy(dtype=float) > 0.5
legal_food = np.where(occupied, -np.inf, food_matrix)
has_free_neighbour = (~occupied).any(axis=1)
best_free_food = legal_food.max(axis=1)
threshold = rational['food_here'].to_numpy(dtype=float) + (
    rational['energy'].to_numpy(dtype=float) * MOVE_COST_FRACTION
)
should_move_alpha1 = has_free_neighbour & (best_free_food * ALPHA_RATIONAL > threshold)

chosen_action = rational['move'].str.strip().to_numpy()
action_to_idx = {action: idx for idx, action in enumerate(dir_names)}
chosen_idx = np.array([action_to_idx.get(action, -1) for action in chosen_action], dtype=int)
chose_direction = chosen_idx >= 0
chosen_best_free = np.zeros(len(rational), dtype=bool)
chosen_rows = np.flatnonzero(chose_direction)
chosen_best_free[chosen_rows] = (
    ~occupied[chosen_rows, chosen_idx[chosen_rows]]
    & np.isclose(
        food_matrix[chosen_rows, chosen_idx[chosen_rows]],
        best_free_food[chosen_rows],
        atol=1e-9,
    )
)

rational['best_free_food'] = best_free_food
rational['alpha1_threshold'] = threshold
rational['should_move_alpha1'] = should_move_alpha1
rational['chosen_best_free'] = chosen_best_free
rational['rational_choice'] = np.where(
    should_move_alpha1,
    chosen_best_free,
    chosen_action == 'Stay',
)

n_rational = int(rational['rational_choice'].sum())
n_choices = len(rational)
p_rational = rational['rational_choice'].mean()

if n_choices == 0:
    print('No decision data available.')
else:

    fig, axes = plt.subplots(len(MODEL_ORDER), 2, figsize=(13, 4.8 * len(MODEL_ORDER)), squeeze=False)
    by_scenario = rational.groupby(['model_label', 'scenario', 'prompt_variant'])['rational_choice'].agg(['sum', 'count', 'mean']).reset_index()
    by_scenario.columns = ['model_label', 'scenario', 'prompt_variant', 'n_rational', 'n_choices', 'P_rational']
    for row, model_label in enumerate(MODEL_ORDER):
        model_rational = rational[rational['model_label'] == model_label]
        p_model = model_rational['rational_choice'].mean()
        ax = axes[row, 0]
        bars = ax.bar(['Rational choice', 'Other choice'], [p_model, 1 - p_model],
                      color=[PLASMA(0.72), PLASMA(0.12)], edgecolor='white', zorder=3)
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                    f'{bar.get_height():.1%}', ha='center', va='bottom', fontsize=11)
        ax.set_ylabel('Proportion')
        ax.set_title(f'{model_label}: choices consistent with rule at α=1')
        ax.set_ylim(0, 1)
        sns.despine(ax=ax)
        ax2 = axes[row, 1]
        sns.barplot(data=by_scenario[by_scenario['model_label'] == model_label], x='scenario', y='P_rational',
                    hue='prompt_variant', hue_order=PROMPT_ORDER, palette=PROMPT_COLORS, edgecolor='white', ax=ax2)
        ax2.set_ylabel(r'$P(\mathrm{choice\ consistent\ with}\ \alpha=1)$')
        ax2.set_title(f'{model_label}: by scenario')
        ax2.set_ylim(0, 1)
        ax2.legend(title='System prompt', frameon=False)
        sns.despine(ax=ax2)

    plt.tight_layout()
    plt.show()

    display(by_scenario.style.format({'n_rational': '{:,.0f}', 'n_choices': '{:,.0f}', 'P_rational': '{:.2%}'}))


## 5. Stay probability vs. Food in current cell

$P(\text{Stay} \mid \text{food\_here} \in \text{bin})$

**Expectation**: agents should be more likely to stay when there is plenty of food in their cell.


In [ ]:
df['is_stay'] = df['move'].str.strip() == 'Stay'
n_bins = 10
df['food_here_bin'] = pd.cut(df['food_here'], bins=n_bins)

stay_food = df.groupby(['model_label', 'prompt_variant', 'food_here_bin'], observed=True)['is_stay'].agg(['mean', 'count']).reset_index()
stay_food.columns = ['model_label', 'prompt_variant', 'food_here_bin', 'P_stay', 'n']
stay_food['bin_center'] = stay_food['food_here_bin'].apply(lambda x: x.mid)

fig, axes = plt.subplots(len(MODEL_ORDER), 1, figsize=(10, 4.8 * len(MODEL_ORDER)), squeeze=False, sharex=True, sharey=True)
for row, model_label in enumerate(MODEL_ORDER):
    ax = axes[row, 0]
    model_data = stay_food[stay_food['model_label'] == model_label]
    for variant in PROMPT_ORDER:
        subset = model_data[model_data['prompt_variant'] == variant]
        ax.plot(subset['bin_center'], subset['P_stay'], 's-', label=variant,
                color=PROMPT_COLORS[variant], markersize=7, linewidth=2)
    ax.set_xlabel(r'$f_{\mathrm{here}}$ (food in current cell)')
    ax.set_ylabel(r'$P(\mathrm{Stay})$')
    ax.set_title(f'{model_label}: ' + r'$P(\mathrm{Stay} \mid f_{\mathrm{here}} \in \mathrm{bin})$')
    ax.legend(title='System prompt', frameon=False)
    sns.despine(ax=ax)
plt.tight_layout()
plt.show()

display(stay_food[['food_here_bin', 'P_stay', 'n']].style.format({'P_stay': '{:.4f}', 'n': '{:,}'}))
df.drop(columns=['is_stay', 'food_here_bin'], inplace=True)


## 6. Action distribution by energy level

How does the action mix (N/S/E/W/Stay) shift as a function of agent energy?

**Expectation**: more "Stay" at low energy (conservative), more movement at high energy.


In [ ]:
df['energy_bin'] = energy_bins(df['energy'])

action_energy = df.groupby(['model_label', 'prompt_variant', 'energy_bin', 'move'], observed=True).size().reset_index(name='count')
totals = df.groupby(['model_label', 'prompt_variant', 'energy_bin'], observed=True).size().reset_index(name='total')
action_energy = action_energy.merge(totals, on=['model_label', 'prompt_variant', 'energy_bin'])
action_energy['freq'] = action_energy['count'] / action_energy['total']

fig, axes = plt.subplots(len(MODEL_ORDER), len(PROMPT_ORDER),
                         figsize=(12 * len(PROMPT_ORDER), 5.5 * len(MODEL_ORDER)), squeeze=False)
for row, model_label in enumerate(MODEL_ORDER):
  for vi, variant in enumerate(PROMPT_ORDER):
    ax = axes[row, vi]
    variant_data = action_energy[(action_energy['model_label'] == model_label) & (action_energy['prompt_variant'] == variant)]
    x = np.arange(len(ENERGY_LABELS))
    width = 0.15
    for j, action in enumerate(ACTION_NAMES):
        subset = variant_data[variant_data['move'] == action].set_index('energy_bin')
        vals = [float(subset.loc[eb, 'freq']) if eb in subset.index else 0.0 for eb in ENERGY_LABELS]
        ax.bar(x + j * width, vals, width, label=action, color=ACTION_COLORS[action], edgecolor='white')
    ax.set_xlabel('Energy bin')
    ax.set_ylabel('Action frequency' if vi == 0 else '')
    ax.set_title(f'{model_label} — {variant}')
    ax.set_xticks(x + width * 2)
    ax.set_xticklabels(ENERGY_LABELS, rotation=15, fontsize=9)
    ax.legend(title='Action', frameon=False)
    sns.despine(ax=ax)
plt.suptitle('Action distribution by energy level', y=1.02)
plt.tight_layout()
plt.show()

df.drop(columns='energy_bin', inplace=True)


## 7. Reproduction threshold estimation

Statistical analysis of the energy distribution of agents that **did** reproduce vs. those that **did not**.


In [ ]:
reproducers = df[df['did_reproduce']]
non_reproducers = df[~df['did_reproduce']]

stats = pd.DataFrame({
    'Metric': [
        'Total decisions',
        'Overall reproduction rate',
        'Mean energy (reproduced)',
        'Mean energy (did not reproduce)',
        'Median energy (reproduced)',
        'Min energy (reproduced)',
        'P10 energy (reproduced)',
        'P25 energy (reproduced)',
        'Mean free cells (reproduced)',
        'Mean free cells (did not reproduce)',
    ],
    'Value': [
        f'{len(df):,}',
        f'{df["did_reproduce"].mean():.4f}',
        f'{reproducers["energy"].mean():.3f}' if len(reproducers) > 0 else 'N/A',
        f'{non_reproducers["energy"].mean():.3f}' if len(non_reproducers) > 0 else 'N/A',
        f'{reproducers["energy"].median():.3f}' if len(reproducers) > 0 else 'N/A',
        f'{reproducers["energy"].min():.3f}' if len(reproducers) > 0 else 'N/A',
        f'{reproducers["energy"].quantile(0.10):.3f}' if len(reproducers) > 0 else 'N/A',
        f'{reproducers["energy"].quantile(0.25):.3f}' if len(reproducers) > 0 else 'N/A',
        f'{reproducers["free_cells"].mean():.3f}' if len(reproducers) > 0 else 'N/A',
        f'{non_reproducers["free_cells"].mean():.3f}' if len(non_reproducers) > 0 else 'N/A',
    ],
})
display(stats.style.hide(axis='index'))

# Distribution plots
if len(reproducers) > 0 and len(non_reproducers) > 0:
    fig, axes = plt.subplots(len(MODEL_ORDER), 2, figsize=(14, 4.8 * len(MODEL_ORDER)), squeeze=False)
    for row, model_label in enumerate(MODEL_ORDER):
        model_df = df[df['model_label'] == model_label]
        model_repro = model_df[model_df['did_reproduce']]
        model_non_repro = model_df[~model_df['did_reproduce']]
        ax = axes[row, 0]
        ax.hist(model_non_repro['energy'], bins=40, alpha=0.65, color=PLASMA(0.12),
                label='Did not reproduce', density=True, zorder=2)
        ax.hist(model_repro['energy'], bins=40, alpha=0.75, color=PLASMA(0.78),
                label='Reproduced', density=True, zorder=3)
        ax.set_xlabel('Energy')
        ax.set_ylabel('Density')
        ax.set_title(f'{model_label}: energy distribution')
        ax.legend()
        sns.despine(ax=ax)
        ax2 = axes[row, 1]
        for fc in sorted(model_df['free_cells'].unique()):
            subset = model_df[model_df['free_cells'] == fc]
            if len(subset) > 50:
                bins = energy_bins(subset['energy'])
                rates = subset.groupby(bins, observed=True)['did_reproduce'].mean()
                ax2.plot(np.arange(len(rates)), rates.values, 'o-', label=f'fc={fc}',
                         color=FREE_CELL_COLORS.get(fc, PLASMA(0.5)), markersize=5, linewidth=1.5)
        ax2.set_xlabel('Energy')
        ax2.set_xticks(np.arange(len(ENERGY_LABELS)))
        ax2.set_xticklabels(ENERGY_LABELS)
        ax2.set_ylabel(r'$P(\mathrm{reproduce})$')
        ax2.set_title(f'{model_label}: ' + r'$P(\mathrm{reproduce} \mid E)$ by free cells')
        ax2.legend(title='Free cells', fontsize=9)
        sns.despine(ax=ax2)

    plt.tight_layout()
    plt.show()


## 8. Per-scenario summary

Overview of decision patterns in each scenario.


In [ ]:
summary_rows = []
for (model_label, scenario, prompt_variant), subset in df.groupby(['model_label', 'scenario', 'prompt_variant']):
    summary_rows.append({
        'Model': model_label,
        'Scenario': scenario,
        'System prompt': prompt_variant,
        'Decisions': len(subset),
        'Reproduction rate': subset['did_reproduce'].mean(),
        'Movement rate': subset['did_move'].mean(),
        'Mean energy': subset['energy'].mean(),
        'Mean free cells': subset['free_cells'].mean(),
        'Stay rate': (subset['move'].str.strip() == 'Stay').mean(),
        'P(repro|fc>0)': subset[subset['free_cells'] > 0]['did_reproduce'].mean() if (subset['free_cells'] > 0).any() else np.nan,
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df.style.format({
    'Decisions': '{:,}',
    'Reproduction rate': '{:.4f}',
    'Movement rate': '{:.4f}',
    'Mean energy': '{:.2f}',
    'Mean free cells': '{:.2f}',
    'Stay rate': '{:.4f}',
    'P(repro|fc>0)': '{:.4f}',
}))


In [ ]:
scenarios = sorted(df['scenario'].unique())
n_scenarios = len(scenarios)

prompt_variants = PROMPT_ORDER
if n_scenarios > 0:
    fig, axes = plt.subplots(len(MODEL_ORDER) * len(prompt_variants), n_scenarios,
                             figsize=(4 * n_scenarios, 4.5 * len(MODEL_ORDER) * len(prompt_variants)), squeeze=False)
    for model_idx, model_label in enumerate(MODEL_ORDER):
     for variant_idx, prompt_variant in enumerate(prompt_variants):
      row_idx = model_idx * len(prompt_variants) + variant_idx
      for idx, scenario in enumerate(scenarios):
        ax = axes[row_idx, idx]
        subset = df[(df['model_label'] == model_label) & (df['scenario'] == scenario) & (df['prompt_variant'] == prompt_variant)]
        if subset.empty:
            ax.set_axis_off()
            ax.set_title(f'{model_label} — {scenario} — {prompt_variant} (no data)', fontweight='bold')
            continue
        subset = subset.copy()
        subset['energy_bin'] = energy_bins(subset['energy'])
        pivot = subset.groupby(['energy_bin', 'free_cells'], observed=True)['did_reproduce'].mean().reset_index()
        pivot_table = pivot.pivot(index='energy_bin', columns='free_cells', values='did_reproduce')
        sns.heatmap(pivot_table, annot=True, fmt='.2f', cmap='YlOrRd',
                    linewidths=0.5, vmin=0, vmax=pivot_table.max().max() * 1.1 if pivot_table.max().max() > 0 else 1,
                    cbar=idx == n_scenarios - 1,
                    cbar_kws={'label': r'$P(\mathrm{repro})$'} if idx == n_scenarios - 1 else {},
                    ax=ax)
        ax.set_title(f'{model_label} — {scenario} — {prompt_variant}', fontweight='bold')
        ax.set_xlabel('Free cells' if idx == n_scenarios // 2 else '')
        ax.set_ylabel('Energy' if idx == 0 else '')

    plt.suptitle(r'$P(\mathrm{reproduce} \mid E,\, \mathrm{free\_cells})$ per scenario',
                 fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


## 9. Temporal evolution of decisions

How do the decision patterns evolve over simulation steps?


In [ ]:
temporal_metrics = [
    ('did_reproduce', r'$P(\mathrm{reproduce})$', 'Reproduction rate'),
    ('did_move', r'$P(\mathrm{move})$', 'Movement rate'),
    ('energy', 'Mean energy', 'Mean agent energy'),
    ('is_stay', r'$P(\mathrm{Stay})$', 'Stay rate'),
]
temporal_df = df.assign(is_stay=df['move'].str.strip().eq('Stay'))
fig, axes = plt.subplots(2 * len(MODEL_ORDER), 2, figsize=(14, 9 * len(MODEL_ORDER)), squeeze=False)
for model_idx, model_label in enumerate(MODEL_ORDER):
  for metric_idx, (metric, ylabel, title) in enumerate(temporal_metrics):
    ax = axes[2 * model_idx + metric_idx // 2, metric_idx % 2]
    for scenario in sorted(df['scenario'].unique()):
      for prompt_variant in PROMPT_ORDER:
        subset = temporal_df[(temporal_df['model_label'] == model_label) &
                             (temporal_df['scenario'] == scenario) &
                             (temporal_df['prompt_variant'] == prompt_variant)]
        temporal = subset.groupby('step')[metric].mean().rolling(window=10, min_periods=1).mean()
        ax.plot(temporal.index, temporal.values, label=f'{scenario} | {prompt_variant}',
                color=SCENARIO_PALETTE.get(scenario, '#888'),
                linestyle='--' if prompt_variant == 'No stated goal' else '-', linewidth=1.5)
    ax.set_xlabel('Step')
    ax.set_ylabel(ylabel)
    ax.set_title(f'{model_label}: {title} over time (smoothed)')
    ax.legend(fontsize=8)
    sns.despine(ax=ax)
plt.suptitle('Temporal evolution of LLM decision patterns', fontweight='bold', y=1.002)
plt.tight_layout()
plt.show()

# ── Explicit-goal vs no-goal system-prompt comparison ──
comparison = df.groupby(['model_label', 'prompt_variant']).agg(
    decisions=('move', 'size'),
    reproduction_rate=('did_reproduce', 'mean'),
    movement_rate=('did_move', 'mean'),
    mean_energy=('energy', 'mean'),
).reset_index()
comparison['stay_rate'] = temporal_df.groupby(['model_label', 'prompt_variant'])['is_stay'].mean().to_numpy()
comparison['rational_choice_rate'] = rational.groupby(['model_label', 'prompt_variant'])['rational_choice'].mean().to_numpy()
display(comparison.style.format({
    'decisions': '{:,}', 'reproduction_rate': '{:.2%}', 'movement_rate': '{:.2%}',
    'stay_rate': '{:.2%}', 'rational_choice_rate': '{:.2%}', 'mean_energy': '{:.2f}',
}))

metric_cols = ['reproduction_rate', 'movement_rate', 'stay_rate', 'rational_choice_rate']
metric_labels = ['Reproduction', 'Movement', 'Stay', r'Rule-consistent ($\alpha=1$)']
fig, axes = plt.subplots(len(MODEL_ORDER), 1, figsize=(10, 4.8 * len(MODEL_ORDER)), squeeze=False, sharex=True, sharey=True)
x = np.arange(len(metric_cols))
width = 0.34
for model_idx, model_label in enumerate(MODEL_ORDER):
    ax = axes[model_idx, 0]
    model_comparison = comparison[comparison['model_label'] == model_label]
    for i, variant in enumerate(PROMPT_ORDER):
        rows = model_comparison[model_comparison['prompt_variant'] == variant]
        if rows.empty: continue
        values = [rows.iloc[0][c] for c in metric_cols]
        ax.bar(x + (i - (len(PROMPT_ORDER) - 1) / 2) * width, values, width,
               label=variant, color=PROMPT_COLORS[variant])
    ax.set_xticks(x)
    ax.set_xticklabels(metric_labels)
    ax.set_ylabel('Rate')
    ax.set_ylim(0, 1)
    ax.set_title(f'{model_label}: system-prompt treatment comparison')
    ax.legend(frameon=False)
    sns.despine(ax=ax)
plt.tight_layout()
plt.show()
